# Suite CPRS — Parsing C into the store

`C.Parse.parse(text, path)` reads C source with tree-sitter-c and gives the callback of the transaction that writes
it: one upsert, binding the workspace and the translation unit, that files the unit in the workspace by its path and
writes the unit's `body` as one value. The whole tree is values: each declaration and statement is a value object, a
branch of `Programs.C.Statement` tagged by its kind in lower case, with the line it begins on (`line`), holding the
statements of its blocks in its properties, in order. Expressions are mbse-expressions' `C` terms, a declaration's
specifiers, declarators and attributes `C Types`' helpers and directives `C Preprocessor` terms, each keeping the
concrete syntax the dialect holds; where a term holds statements or declarations, its `block`, `members` or `group`
holds them. Each case parses constructs into one store, reads the unit's tree back as plain data within a transaction
(a term as `ToText` writes it, or as its kind, attributes and subterms; a helper as the printer writes it), and checks
it; then invalid syntax, which is kept, and the one error, text that is no text; then each place where tree-sitter-c
departs from C, by the number `Parse` gives it. Its cases port 0.3's C cases of PRS (`03_Parse.ipynb`).

In [ ]:
import itertools
import json
from pathlib import Path

from mbse.Expressions.C.Expressions import Expressions as CExpressions
from mbse.Expressions.C.Expressions.Text import ToText
from mbse.Expressions.Framework import Objects, Terms
from mbse.Programs import C
from mbse.Programs.C import Check, Parse, Print
from mbse.Programs.C.Statements import KINDS, PARTS
from mbse.Programs.Framework import Model
from mbse.Programs.Framework.Errors import ParseError
from mbse.Schemas.Framework import Plain, References, Schemas
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = C.store()
PATHS = (f"case{n}.c" for n in itertools.count())
HOLDERS = ("block", "members", "group")  # the properties of terms and helpers that hold this grammar's statements


def shape(term):
    """A term as plain data: its kind, its attributes (a helper as its kind and attributes), and its subterms."""
    def part(value):
        if isinstance(value, Terms.Part):
            return (value.kind, {k: part(v) for k, v in value.attributes.items()})
        return [part(v) for v in value] if isinstance(value, tuple) else value
    tag, record = Terms.kind_of(term)
    attributes = {k: part(v) for k, v in Terms.attributes_of(record).items()}
    held = Objects.properties(record)
    attributes.update({name: plain(held[name]) for name in HOLDERS if name in held})
    return (tag, attributes, [None if a is None else shape(a) for a in Terms.arguments_of(record)])


def plain(value, shapes=False):
    """What a property holds, as plain data: a term as its text (or its `shape`), a helper of `C Types` as the printer
    writes it, a container as a list, a union's value as its branch's, a value object as a dict of its kind and its
    properties."""
    if isinstance(value, Objects.Listed):
        return [plain(v, shapes) for v in value.values]
    if Terms.is_term(value):
        return shape(value) if shapes else ToText(value)
    if Objects.is_object(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        record = Terms.kind_of(value)[1]  # a branch of this grammar's, a term (a `for`'s initializer), or a helper
        if Terms.is_term(record) or record.as_schema().name.startswith("Programs."):
            return plain(record, shapes)
        return Print.Printer().written(value)
    if Objects.is_object(value) and value.as_schema().name.startswith("Expressions."):
        return Print.Printer().written(value)
    if Objects.is_object(value):
        return {"kind": value.as_schema().name.rsplit(".", 1)[-1]} | {
            k: plain(v, shapes) for k, v in Objects.properties(value).items()}
    return value


def unit(store, path):
    """The translation unit at `path`, read within the transaction open on `store`."""
    return store.singleton(Model.WORKSPACE).files({"path": path})


def parse(text, shapes=False):
    """The declarations of the translation unit `text` parses to, written in the store at a path of its own, read
    back."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    with STORE.transaction():
        return plain(Objects.properties(unit(STORE, path))["body"], shapes)


def first(text, shapes=False):
    return parse(text, shapes)[0]


def body(text, shapes=False):
    """The statements of `text`, the body of a function `f`."""
    return first("void f(void) {\n" + text + "\n}", shapes)["body"]["body"]


def statement(text, shapes=False):
    return body(text, shapes)[0]


def value(text, shapes=False):
    """The expression of the expression statement `text`, within a function."""
    return statement(text + ";", shapes)["expression"]


def initial(text, shapes=False):
    """What initializes the one variable `text` declares."""
    return first(text, shapes)["declarators"][0]["initializer"]


def kinds(nodes):
    return [n["kind"] for n in nodes]


def printed(text):
    """The text the translation unit `text` parses to prints as."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return Print.print_module(STORE, path)


def same(text):
    """Whether `text`, in the printer's layout, prints back as written."""
    return printed(text) == text


def assigned(store, frame):
    """What `frame`, given back by a transaction on `store`, assigns, as `References.OfFrame` writes it as plain data."""
    written = Plain.ToPlain(store).Value(store.registered("References.OfFrame"), References.of(store, frame))
    return written["assigns"]

## CPRS-01 · The conformance source is one transaction: every statement, declaration and member kind and every part, terms the dialect validates, and JSON that reads back and writes the same bytes

In [ ]:
def walk(value, outermost=True):
    """The value objects and the outermost terms within `value`, in source order: a term's subterms and helpers are its
    own, but the statements its block, members or group holds are walked as any."""
    if isinstance(value, Objects.Listed):
        for v in value.values:
            yield from walk(v, outermost)
    elif Terms.is_term(value):
        if outermost:
            yield value
        for name, held in Objects.properties(Terms.kind_of(value)[1]).items():
            yield from walk(held, name in HOLDERS)
    elif Objects.is_object(value) and isinstance(value.as_schema(), Schemas.OfUnion.Data):
        yield from walk(Terms.kind_of(value)[1], outermost)
    elif Objects.is_object(value):
        if outermost:
            yield value
        for name, held in Objects.properties(value).items():
            yield from walk(held, outermost or name in HOLDERS)


name = "c23.c"
text = (SOURCES / name).read_text(encoding="utf-8")
(frame,) = STORE.transaction(Parse.parse(text, name))
assert sorted(frame) == ["unit", "ws"]  # the workspace and the unit; every declaration is a value within it
(assign,) = assigned(STORE, frame)  # the unit, given its whole tree as one nested value
assert len(assign["properties"]["body"]["Programs.C.Body"]) == 73
seen, problems = set(), []
with STORE.transaction():
    for node in walk(Objects.properties(unit(STORE, name))["body"]):
        if Terms.is_term(node):  # a literal's type is read under an ABI, which only an expression's holder names
            problems.extend(p for p in CExpressions.DIALECT.validate(node) if "names no ABI" not in p and "is not bound" not in p)
        else:
            seen.add(node.as_schema().name.rsplit(".", 1)[-1])
assert set(KINDS) <= seen, set(KINDS) - seen  # every statement, declaration and member kind
assert set(PARTS) <= seen, set(PARTS) - seen  # and every part
assert "Error" not in seen and problems == [], problems[:5]
# Written as plain JSON, read into a fresh store holding the installs, executed, and written again: the same bytes
written = json.dumps(Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(Parse.parse(text, name, STORE))))
fresh = C.store()
(again,) = fresh.transaction(Plain.FromPlain(fresh).Transaction(json.loads(written)))
assert sorted(again) == sorted(frame) and assigned(fresh, again) == assigned(STORE, frame)
assert json.dumps(Plain.ToPlain(fresh).Transaction(fresh.Mutations.OfTransaction(
    Parse.parse(text, name, fresh)))) == written
print(len(seen), "kinds")

## CPRS-02 · The transaction: one upsert by the file's path, binding the workspace and the unit, the unit filed by its path, its tree one nested value, each statement in its parent's property, in order, with its line, a statement expression's statements in its `block`

In [ ]:
source = "int f(int a) {\n    if (a) {\n    } else {\n        a = 1;\n    }\n    return ({\n        a;\n    });\n}\n"
(frame,) = STORE.transaction(Parse.parse(source, "shape.c"))
assert sorted(frame) == ["unit", "ws"]
assert set(frame.assigns) == {"unit"} and set(frame.assigns["unit"]) == {"body"}
(assign,) = assigned(STORE, frame)
one = {"literal": {"value": {"int": 1}, "text": "1", "base": 10}}
a = {"variable": {"name": "a"}}
assert assign["properties"]["body"]["Programs.C.Body"][0]["functiondefinition"]["body"] == {"line": 1, "body": [
    {"ifstatement": {"line": 2, "condition": a, "consequence": {"compoundstatement": {"line": 2}},
                     "alternative": {"compoundstatement": {"line": 3, "body": [{"expressionstatement": {
                         "line": 4, "expression": {"assignmentexpression": {
                             "left": a, "operator": "=", "right": one}}}}]}}}},
    {"returnstatement": {"line": 6, "value": {"statementexpression": {"block": {"compoundstatement": {
        "line": 6, "body": [{"expressionstatement": {"line": 7, "expression": a}}]}}}}}}]}
with STORE.transaction():
    file = unit(STORE, "shape.c")
    assert file.as_schema().name == "Programs.C.TranslationUnit"
    (definition,) = Objects.properties(file)["body"].values
    tag, node = Terms.kind_of(definition)
    assert (tag, node.as_schema().name) == ("functiondefinition", "Programs.C.FunctionDefinition")
    _, returned = Terms.kind_of(Objects.properties(Objects.properties(node)["body"])["body"].values[1])
    # a statement expression, a term, holds its statements in its `block`: this grammar's `CompoundStatement`, a
    # branch of the dialect's open union `Expressions.C.Expressions.Block`
    block = Objects.properties(Terms.kind_of(Objects.properties(returned)["value"])[1])["block"]
    assert block.as_schema().name == "Expressions.C.Expressions.Block"
    tag, node = Terms.kind_of(block)
    assert (tag, node.as_schema().name) == ("compoundstatement", "Programs.C.CompoundStatement")
# Run again, the upsert finds what it wrote
STORE.transaction(Parse.parse(source, "shape.c"))
with STORE.transaction():
    assert len(Objects.properties(unit(STORE, "shape.c"))["body"].values) == 1
# The callback names its schemas through what store.transaction gives it; a bare builder needs the store
with raises(TypeError):
    STORE.Mutations.OfTransaction(Parse.parse("int x;", "bare.c"))
built = Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(Parse.parse("int x;", "bare.c", STORE)))
assert "bare.c" in json.dumps(built)
assert parse("") == [] and kinds(parse("int x;\n\n\nint y;\n")) == ["SimpleDeclaration", "SimpleDeclaration"]
assert [s["line"] for s in parse("int x;\n\n[[deprecated]]\nint y;\n")] == [1, 3]  # where its attributes start
print("ok")

## CPRS-03 · Literals keep their spelling: integers in each base with their suffixes and separators, floating literals, characters and strings with their prefixes and escapes, `true`, `false`, `nullptr`, and adjacent strings

In [ ]:
def literal(text):
    tag, attributes, _ = value(text, shapes=True)
    return tag, attributes


assert literal("1'000ull") == ("literal", {"value": 1000, "text": "1'000ull", "base": 10, "separators": True,
                                           "suffix": "ull"})
assert [literal(t)[1]["base"] for t in ("0x1F", "0b101", "017", "0", "10")] == [16, 2, 8, 8, 10]
assert [literal(t)[1]["value"] for t in ("0x1F", "0b101", "017", "0", "10")] == [31, 5, 15, 0, 10]
assert literal("1wb")[1]["suffix"] == "wb" and literal("0xFFu")[1] == {"value": 255, "text": "0xFFu", "base": 16,
                                                                     "suffix": "u"}
assert literal("1.5e-3") == ("literal", {"value": 0.0015, "text": "1.5e-3"})  # no integer: no base
assert literal("0x1.8p3")[1]["value"] == 12.0 and literal(".5")[1]["value"] == 0.5 and literal("5.")[1]["value"] == 5
assert literal("1.0f")[1]["suffix"] == "f" and literal("1.5dd")[1]["suffix"] == "dd"
assert literal("1e999")[1]["value"] == float("inf") == literal("0x1p99999")[1]["value"]  # beyond the largest
assert literal("'a'") == ("literal", {"value": 97, "text": "'a'", "quote": "'"})
assert literal("L'w'")[1] == {"value": 119, "text": "L'w'", "prefix": "L", "quote": "'"}
assert literal("u8'x'")[1]["prefix"] == "u8" and literal("'ab'")[1]["value"] == 0x6162  # as GCC reads it
assert literal(r"'\n'")[1]["value"] == 10 and literal(r"'\x41'")[1]["value"] == 65
assert literal(r"'\101'")[1]["value"] == 65 and literal(r"'\''")[1]["value"] == 39
assert literal(r'"a\tbé\U0001F600\e\?\q"')[1]["value"] == "a\tbé\U0001F600\x1b?q"
assert literal('"a\\\nb"')[1]["value"] == "ab"  # a line continued
assert literal('u"s"')[1] == {"value": "s", "text": 'u"s"', "prefix": "u", "quote": '"'}
assert literal("true") == ("literal", {"value": True, "text": "true"})
assert literal("false") == ("literal", {"value": False, "text": "false"})
assert literal("nullptr") == ("constant", {"singleton": "nullptr", "text": "nullptr"})
concatenated = value('"a" L"b" PRIu64', shapes=True)
assert concatenated[0] == "concatenatedstring" and [p[0] for p in concatenated[2]] == ["literal", "literal",
                                                                                         "variable"]
for text in ("1'000ull", "0x1F", "0b101", "017", "1.5e-3", ".5", "5.", "0x1.8p3", "'a'", "L'w'", r"'\n'",
             '"a\\tb"', "true", "nullptr", '"a" L"b" PRIu64'):
    assert value(text) == text, (text, value(text))
print("ok")

## CPRS-04 · Expressions: every operator, with C's grouping; calls, members, subscripts, casts, `sizeof`, `alignof`, `offsetof`, `_Generic`, compound literals, designated initializers and GNU's statement and `__extension__` expressions

In [ ]:
def tree(text):
    return value(text, shapes=True)


assert tree("a, b, c")[2][0][1] == {"operator": ","}  # the comma groups to the left
assert tree("a = b += c")[2][1][1] == {"operator": "+="}
assert tree("a ? b : c")[0] == "conditionalexpression" and tree("a ?: c")[2][1] is None  # GNU's
assert tree("-a")[1] == {"operator": "-"} and tree("*p")[1] == {"operator": "*"} and tree("&x")[1] == {"operator": "&"}
assert tree("x++")[:2] == ("postfixexpression", {"operator": "++"}) and tree("--x")[:2] == (
    "unaryexpression", {"operator": "--"})
assert tree("f(a, b)")[0] == "callexpression" and len(tree("f(a, b)")[2]) == 3
assert tree("s->x")[1] == {"operator": "->", "member": ("identifier", {"spelling": "x"})}
assert tree("a[1]")[0] == "subscriptexpression"
assert tree("(int)x")[2][0][0] == "typeid" and tree("sizeof(int)")[2][0][0] == "typeid"
assert tree("sizeof x")[2][0] == ("variable", {"name": "x"}, [])
assert tree("_Alignof(int)")[1] == {"keyword": "_Alignof"} and tree("alignof(int)")[1] == {"keyword": "alignof"}
assert tree("offsetof(struct s, a)")[2][0] == ("variable", {"name": "offsetof"}, [])
generic = tree("_Generic(a, int: 1, default: 2)")
assert generic[0] == "genericselection" and generic[2][1][0] == "typeid" and generic[2][3] is None
compound = tree("(struct p){1, 2}")
assert compound[0] == "compoundliteralexpression" and compound[1]["initializer"][0] == "initializerlist"
designated = first("int a[] = {[1] = 2, [3 ... 4] = 5, .x = 6, .y.z = 7, 8,};", shapes=True)["declarators"][0][
    "initializer"]
assert designated[1] == {"trailing_comma": True} and [i[0] for i in designated[2]] == [
    "designatedinitializer"] * 4 + ["literal"]
assert tree("({ int t = 1; t; })")[1]["block"]["body"][1]["expression"] == "t"
assert tree("__extension__ 1")[0] == "extensionexpression" and tree("(a)")[0] == "parenthesizedexpression"
for text in ("a + b * c", "(a + b) * c", "a = b ? c : d", "*p++", "s->x.y[1]", "(int)3.5", "sizeof(int*)",
             "f(a, b)", "!a && b || c", "a << 1 | b & c ^ d", "x = (struct p){1, 2}"):
    assert value(text) == text, (text, value(text))
print("ok")

## CPRS-05 · Declarations: specifiers and attributes in source order, declarators inside out, initializers, typedefs, function definitions, K&R definitions and `static_assert`

In [ ]:
declaration = first("[[nodiscard]] static inline const unsigned long long int [[a]] *x = 0, y = {1}, z;")
assert declaration["attributes"] == ["[[nodiscard]]"]
assert declaration["specifiers"] == ["static", "inline", "const", "unsigned", "long", "long", "int", "[[a]]"]
assert [d["declarator"] for d in declaration["declarators"]] == ["*x", "y", "z"]
assert [d.get("initializer") for d in declaration["declarators"]] == ["0", "{1}", None]
declarator = first("int (*f)(int);", shapes=True)["declarators"][0]["declarator"]
assert declarator == "(*f)(int)"
assert first("size_t n;")["specifiers"] == ["size_t"] and first("typedef int T, *P;")["specifiers"] == [
    "typedef", "int"]
assert same("int *a[3], (*b)[3], **c, *const f = 0, *g(int), (*h(int))(char);\n")
assert same("void (*handlers[4])(int);\n") and same("void f(int[*], int[static 3], int[const 4]);\n")
assert same("void f(int, ...);\nvoid h(int (*)(int), int (*p)[3]);\n") is False  # an abstract declarator: no space
assert printed("void h(int (*)(int));\n") == "void h(int(*)(int));\n"
assert same("int x [[a]], y [[b]][3];\n") and same("__extension__ typedef long long ll;\n")
assert same("__declspec(dllexport) _Alignas(16) alignas(int) int x;\n")
assert same("int f(void) __attribute__((noinline, format(printf, 1, 2)));\n")
assert first("[[gnu::cold]] void f(void);")["attributes"] == ["[[gnu::cold]]"]
assert same("static_assert(1);\n_Static_assert(1, \"m\");\n")
assert kinds(parse("static_assert(1);")) == ["StaticAssertDeclaration"]
definition = first("int old(a, b)\n    int a;\n    char *b;\n{\n    return a;\n}\n")
assert definition["declarator"] == "old(a, b)" and [d["declarators"][0]["declarator"] for d in definition[
    "declarations"]] == ["a", "*b"]
assert same("int old(a, b)\n    int a;\n    char *b;\n{\n    return a;\n}\n")
assert same("int main(int argc, char *argv[]) {\n    return 0;\n}\n")
assert same('asm("nop");\n') and kinds(parse('asm("nop");')) == ["AsmDeclaration"]
print("ok")

## CPRS-06 · Statements, labels and their substatements: a `case` labels the first statement after it, the others following it; a label may end a block (C23) or precede a declaration

In [ ]:
labels = statement("switch (x) { case 1: a(); b(); case 2: case 3: c(); default: ; }")["body"]["body"]
assert kinds(labels) == ["CaseStatement", "ExpressionStatement", "CaseStatement", "DefaultStatement"]
assert labels[2]["statement"]["kind"] == "CaseStatement" and labels[2]["statement"]["statement"]["kind"] == (
    "ExpressionStatement")
assert "statement" not in statement("switch (x) { case 1: }")["body"]["body"][0]
assert statement("l: int x = 1;")["statement"]["kind"] == "SimpleDeclaration"
assert same("void f(void) {\n    if (a)\n        b();\n    else if (c) {\n        d();\n    } else\n        e();\n}\n")
assert same("void f(void) {\n    while (a--)\n        continue;\n    do\n        b();\n    while (c);\n    for (;;)\n"
            "        break;\n    for (x = 0, y = 1; x < y; ++x) {}\n    for (int i = 0; i < n; ++i) {}\n}\n")
assert same("void f(void) {\n    switch (m) {\n        case 1: {\n            break;\n        }\n    }\n}\n")
assert same("void f(void) {\nstart:\n    goto start;\nend:\n    int x;\n    return;\n}\n")
assert same("void f(void) {\n    [[gnu::hot]] if (a) {}\n    [[fallthrough]];\n    {}\n    ;\n}\n")
assert statement("[[fallthrough]];")["kind"] == "AttributeDeclaration"  # a declaration (C23)
assert same('void f(void) {\n    asm volatile inline goto("jmp %l0" : : : : done);\n'
            '    __asm__("mov %1, %0" : "=r" (a) : [in] "r" (b) : "memory", "cc");\n}\n')
asm = statement('asm volatile("x" : [o] "=r" (a) : "r" (b) : "memory");')
assert asm["volatile"] and asm["outputs"][0]["name"] == "o" and asm["clobbers"] == ['"memory"']
initializer = statement("for (i = 0; i < 3; i++) ;")["initializer"]
assert initializer == "i = 0" and statement("for (int i = 0;;) ;")["initializer"]["kind"] == "SimpleDeclaration"
print("ok")

## CPRS-07 · Structs, unions and enumerations: a struct's or a union's member declarations in its `members`, bit-fields, anonymous members, a member's `static_assert`, directives and comments; an enumeration's enumerators, directives and comments, `C Types`' and `C Preprocessor`'s terms

In [ ]:
for text in ("struct s;\nunion u;\nenum e;\n", "struct s {\n    int x, y : 3;\n    int : 0;\n} v;\n",
             "struct s {\n    union {\n        int i;\n        float f;\n    };\n    struct {\n        int c;\n    } n;\n};\n",
             "enum e {\n    A,\n#ifdef B\n    B,\n#endif\n    // c\n    C\n};\n", "enum e {\n    A = 1,\n    B,\n};\n",
             "struct s {\n#ifdef A\n    int a;\n#else\n    int b;\n#endif\n    // c\n    int c; /* d */\n};\n",
             "void f(void) {\n    struct s {\n        int a;\n    };\n    enum e {\n        A\n    };\n}\n"):
    assert same(text), (text, printed(text))
path = next(PATHS)
STORE.transaction(Parse.parse("struct s { int x; int : 0; };", path))
with STORE.transaction():
    (declaration,) = Objects.properties(unit(STORE, path))["body"].values
    (specifier,) = Objects.properties(Terms.kind_of(declaration)[1])["specifiers"].values
    members = Objects.properties(Terms.kind_of(specifier)[1])["members"]
    assert members.as_schema().name == "Expressions.C.Types.Members"
    tag, node = Terms.kind_of(members)
    assert (tag, node.as_schema().name) == ("memberlist", "Programs.C.MemberList")
    named, unnamed = [Terms.kind_of(m)[1] for m in Objects.properties(node)["members"].values]
    assert Objects.properties(named)["line"] == 1 and "declarator" not in plain(unnamed)["declarators"][0]
    (declarator,) = Objects.properties(unnamed)["declarators"].values
    assert sorted(Objects.properties(declarator)) == ["bitfield"]  # unnamed
assert first("enum e : int { A = 1, B };")["specifiers"] == ["enum e : int {\n    A = 1,\n    B\n}"]
print("ok")

## CPRS-08 · Directives are statements where statements and members are listed, holding their `C Preprocessor` term, a conditional's lines its `group`; comments are statements there, on their own lines or trailing; elsewhere they are dropped

In [ ]:
assert same('#include <a>\n#include "b.h"\n#include MACRO\n#include_next <c>\n#define X\n#define Y 1\n'
            "#define Z(a, b) a##b\n#define V(...) f(__VA_ARGS__)\n#define W(a, ...) a\n#pragma once\n#undef X\n"
            '#line 3 "f"\n#error stop\n#embed "data.bin"\n')
directive = first("#include MACRO\n", shapes=True)
assert directive["kind"] == "Directive" and directive["directive"][0] == "includedirective"
assert first("#pragma\nint x;")["directive"] == "#pragma"
assert printed("#define X /* a\n   b */ 1\n") == "#define X 1\n"  # a comment within it, dropped
assert same("#if defined(A) && B > 1\nint a;\n#elif !defined(C)\nint c;\n#else\nint d;\n#endif\n")
assert same("#ifdef A\n#elifdef B\nint b;\n#elifndef C\n#endif\n#ifndef D\n#endif\n")
conditional = first("#ifdef A\nint a;\n#else\nint b;\n#endif\n", shapes=True)["directive"]
assert conditional[0] == "ifdefdirective" and kinds(conditional[1]["group"]["body"]) == ["SimpleDeclaration"]
assert conditional[2][0][1]["group"]["body"][0]["kind"] == "SimpleDeclaration"  # the `#else`'s group
assert same("void f(void) {\n#if X\n    a();\n#endif\n}\n")
assert same("// line\n/* block */\nint x; // trailing\nvoid f(void) {\n    // inside\n}\n")
comments = parse("/* a */\n// b\r\nint x; // c\n")
assert comments[0] == {"kind": "Comment", "line": 1, "text": " a ", "block": True}
assert comments[1] == {"kind": "Comment", "line": 2, "text": " b"}
assert comments[3] == {"kind": "Comment", "line": 3, "text": " c", "trailing": True}
assert printed("void f(void) {\n    switch (x) {\n        case 1: // one\n            a();\n    }\n}\n") == (
    "void f(void) {\n    switch (x) {\n        case 1:\n            a();\n            // one\n    }\n}\n")
assert first("int f(int /* dropped */ a);")["declarators"][0]["declarator"] == "f(int a)"
print("ok")

## CPRS-09 · Invalid syntax is kept, never raised: an `Error` where a statement, a declaration or a member stands, an `error` or `missing` term within an expression, each as written; only text that is no text raises

In [ ]:
mixed = parse("int a = 1;\nint b = ;\nint c;\n")
assert kinds(mixed) == ["SimpleDeclaration", "SimpleDeclaration", "SimpleDeclaration"]
assert first("int b = ;", shapes=True)["declarators"][0]["initializer"] == ("missing", {"kind": "identifier"}, [])
inside = body("a = 1;\nif (a {}\nb = 3;")
assert kinds(inside) == ["ExpressionStatement", "Error", "ExpressionStatement"]
assert inside[1] == {"kind": "Error", "line": 3, "text": "if (a {}"}  # a `MISSING` `)`, which no term holds  # a `MISSING` `)`, which no term holds
recovered = statement("if (a {\n    b;\n    { c; }\n}")
assert recovered["kind"] == "Error" and kinds(recovered["body"]) == ["ExpressionStatement", "CompoundStatement"]
assert printed("struct s { int a; int b c; };") == "struct s {\n    int a;\n    int b c;\n};\n"
assert value("x = = 2", shapes=True)[:2] == ("error", {"text": "x = = 2"})
assert value("(1 + )", shapes=True)[2][0][2][1] == ("missing", {"kind": "identifier"}, [])
assert value("08", shapes=True) == ("error", {"text": "08"}, [])  # a literal C rejects
assert value('x = asm("nop")', shapes=True)[2][1][:2] == ("error", {"text": 'asm("nop")'})  # `asm` is no expression
assert parse('char *s = "😀";\nx = 1;')[1] == {"kind": "Error", "line": 2, "text": "x = 1;"}  # beyond the first plane
assert first("enum e { A, = };", shapes=True)["specifiers"] == ["enum e {\n    A\n    =\n}"]  # an error among them
assert kinds(parse("int f(int a, +);\nint * __sptr p;\nint __based(x) * q;\nint a __attribute__((x)) b;")) == [
    "Error"] * 5  # Microsoft's pointer modifiers; GNU's attributes before what is no declarator's end
assert printed("struct s { int a; + };") == "struct s {\n    int a;\n    +\n};\n"  # among members
assert statement("switch (x) { case 1 2: f(); case 3 }")["body"]["body"] == [
    {"kind": "Error", "line": 2, "text": "case 1 2: f();", "body": [{"kind": "ExpressionStatement", "line": 2,
                                                                    "expression": "f()"}]},
    {"kind": "Error", "line": 2, "text": "case 3"}]  # what labels it is invalid, or its `:` missing
assert parse("[[x]] if (a) {}")[0]["text"] == "[[x]] if (a) {}"  # a statement at file scope, attributed
# What the pre-pass rewrote that is invalid still
assert kinds(parse("typeof() x;\n_BitInt(+) x;\n_Atomic(+) y;\n_Atomic(a + 1) z;")) == ["Error"] * 4
for text in ("_BitInt(3", "typeof(a +", "struct [[a", "struct s __attribute__((a", "int * [[a", "enum e { A [[a",
             "int x __attribute__((a", "switch (x", "switch (x) { case 1 ... 3", 'int x asm("r"', "_Pragma(\"a\" int x;"):  # unclosed: nothing the pre-pass could read back
    assert {kind for kind in kinds(parse(text))} == {"Error"}, text
assert kinds(parse('x) asm("r");')) == ["Error", "AsmDeclaration"]  # a `)` that closes nothing: no declarator's
assert printed("struct s __attribute__((a)) v;\nint b = ;\n") == "struct __attribute__((a)) s v;\nint b = ;\n"
# Kept in the store and in JSON as any statement and term are
written = json.dumps(Plain.ToPlain(STORE).Transaction(STORE.Mutations.OfTransaction(
    Parse.parse("int a;\nx = 1;\n", "invalid.c", STORE))))
fresh = C.store()
fresh.transaction(Plain.FromPlain(fresh).Transaction(json.loads(written)))
with fresh.transaction():
    nodes = [Terms.kind_of(n)[1] for n in Objects.properties(unit(fresh, "invalid.c"))["body"].values]
    assert [node.as_schema().name for node in nodes] == ["Programs.C.SimpleDeclaration", "Programs.C.Error"]
    assert Objects.properties(nodes[1]) == {"line": 2, "text": "x = 1;"}  # a statement at file scope
# Only text that is no text raises, before any transaction
with raises(ParseError, match="not text: a lone surrogate"):
    Parse.parse("int x;\n  \udfff", "failing.c")
try:
    Parse.parse("int x;\n  \udfff", "failing.c")
except ParseError as error:
    assert (error.line, error.column) == (2, 3)
print("ok")

## CPRS-10 · Where tree-sitter-c departs from C: each place `Parse`'s docstring enumerates, by its number, with the tree read and the text printed back

In [ ]:
def specifiers(text):
    return first(text)["specifiers"]


# What tree-sitter-c cannot parse, which the pre-pass rewrites
# 1. `_BitInt(N)`
assert first("unsigned _BitInt(8) x;", shapes=True)["specifiers"] == ["unsigned", "_BitInt(8)"]
assert same("_BitInt(N + 1) a = 1wb;\n")
# 2. `typeof` of an expression
typed = first("typeof(a + 1) x;", shapes=True)
assert typed["specifiers"] == ["typeof(a + 1)"] and same("typeof_unqual(f()) y;\n__typeof__(*p) z;\n")
# 3. `_Atomic(T)`
assert specifiers("_Atomic(int *) a;") == ["_Atomic(int*)"] and same("_Atomic(int) a;\n_Atomic int b;\n")
# 4. Type inference
inferred = statement("auto x = 1;")
assert inferred["specifiers"] == ["auto"] and inferred["declarators"][0]["declarator"] == "x"
assert same("static auto y = 2;\nconstexpr auto z = 3;\n") and statement("auto const x = 1;")["specifiers"] == [
    "auto", "const"]
# 5. Standard attributes after `struct`, `union` or `enum`, and after an enumerator's name
assert specifiers("struct [[deprecated]] s;") == ["struct [[deprecated]] s"]
assert same("enum [[nodiscard]] e {\n    A [[deprecated]] = 1,\n    B\n};\n")
assert printed("struct s __attribute__((packed)) __attribute__((a)) {};\n") == (
    "struct __attribute__((packed)) __attribute__((a)) s {};\n")  # GNU's, before its body: after its keyword
assert first("int * [[a]] p;")["declarators"][0]["declarator"] == "*[[a]] p" and same("int *[[a]] const q;\n")
# 6. GNU attributes after a declarator
attributed = first("int *p __attribute__((unused)) = 0, a[3] __attribute__((aligned(8)));")["declarators"]
assert [d["declarator"] for d in attributed] == ["*p __attribute__((unused))", "a[3] __attribute__((aligned(8)))"]
assert same("int x __attribute__((aligned(16))) = 0;\n")
# 7. `static_assert` among a struct's members
assert same('struct s {\n    int a;\n    static_assert(1, "x");\n};\n')
assert same('struct s {\n#ifdef A\n    static_assert(1);\n#endif\n};\n')  # in a group: once
# 8. An enumeration's underlying type of several specifiers, or a name
assert specifiers("enum e : unsigned char { A };") == ["enum e : unsigned char {\n    A\n}"]
assert same("enum f : uint8_t {\n    B\n};\n")
assert printed("enum e : int {\n    A\n};\nenum f g;\nint b = ;\n") == (  # in a text with an error: a keyword
    "enum e : int {\n    A\n};\nenum f g;\nint b = ;\n")  # alone, or no underlying type
# 9. GNU's case ranges
ranged = statement("switch (x) { case 1 ... 3: break; }")["body"]["body"][0]
assert (ranged["value"], ranged["last"]) == ("1", "3")
conditional = statement("switch (x) { case a ? 1 : 2: f(); case 3 ... 4: ; }")["body"]["body"]
assert [label["value"] for label in conditional] == ["a ? 1 : 2", "3"]  # a conditional's `:` is no label's
# 10. A `switch` whose body is no compound statement, the statements after it following it
switched = body("switch (x)\n    case 1:\n        f();\ng();")
assert kinds(switched) == ["SwitchStatement", "ExpressionStatement"] and switched[0]["body"]["kind"] == (
    "CaseStatement")
assert same("void f(void) {\n    switch (x)\n        case 1:\n            f();\n    g();\n}\n")
# 11. GNU's computed `goto`
assert statement("goto *table[i];") == {"kind": "GotoStatement", "line": 2, "target": "table[i]"}
assert same("void f(void) {\n    goto *p;\n}\n")
# 12. The null directive
assert parse("#\nint x;")[0] == {"kind": "Directive", "line": 1, "directive": "#"} and same("#\n")
# 13. Digraphs
assert printed("int a<:3:> = <% 1 %>;\n%:define X 1\n") == "int a[3] = {1};\n#define X 1\n"
# 14. GNU's asm label before what initializes a declarator, or before GNU attributes
labeled = statement('register int r asm("r1") = 3;', shapes=True)["declarators"][0]
assert labeled == {"kind": "InitDeclarator", "declarator": 'r asm("r1")', "initializer": ("literal", {
    "value": 3, "text": "3", "base": 10}, [])}
assert same('void f(void) {\n    register int r asm("r1") = 3, s __asm__("r2");\n}\n')
assert first('int *x asm("x2") __attribute__((unused)), y;')["declarators"][0]["declarator"] == (
    '*x __attribute__((unused)) asm("x2")')  # the attributes the declarator's (mbse-expressions writes them first)
path = next(PATHS)  # among a struct's members, as tree-sitter reads it
STORE.transaction(Parse.parse('struct s { int a asm("b"); };', path))
assert Check.validate(STORE, path) == ["body[0].specifiers[0].members.members[0]: invalid syntax 'int a asm(\"b\");'"]
# 15. `_Pragma("...")` where an item stands, without `;`
pragmas = parse('_Pragma("once")\nint x;\n_Pragma("a") _Pragma("b") int y;', shapes=True)
assert kinds(pragmas) == ["Directive", "SimpleDeclaration", "Directive", "Directive", "SimpleDeclaration"]
assert pragmas[0] == {"kind": "Directive", "line": 1, "directive": ("pragmaexpression", {}, [
    ("literal", {"value": "once", "text": '"once"', "quote": '"'}, [])])}
assert same('void f(void) {\n    _Pragma("omp parallel")\n    g();\n}\n')
assert same('struct s {\n    _Pragma("pack(1)")\n    int x;\n};\n') and same(
    'enum e {\n    A,\n    _Pragma("x")\n    B\n};\n')
path = next(PATHS)  # among enumerators, the term itself
STORE.transaction(Parse.parse('enum e { A, _Pragma("x") B };', path))
with STORE.transaction():
    (declaration,) = Objects.properties(unit(STORE, path))["body"].values
    (specifier,) = Objects.properties(Terms.kind_of(declaration)[1])["specifiers"].values
    enumerators = Objects.properties(Terms.kind_of(specifier)[1])["enumerators"].values
    assert [Terms.kind_of(e)[0] for e in enumerators] == ["enumerator", "pragmaexpression", "enumerator"]
assert kinds(parse('_Pragma(x) int c;')) == ["Error", "SimpleDeclaration"]  # of no string literal (38)
assert initial('int a[] = { _Pragma("x") 1 };', shapes=True)[0] == "error"  # within an initializer, as read
assert kinds(body('l: _Pragma("b") h();')[0:1]) == ["LabeledStatement"] and body('l: _Pragma("b") h();')[0][
    "statement"]["kind"] == "Error"  # after a label, as read
# What tree-sitter-c misreads, which the pre-pass rewrites
# 16. `_Thread_local`
assert specifiers("_Thread_local int a;") == ["_Thread_local", "int"] and same("static _Thread_local int b;\n")
# 17. `_Complex` and `_Imaginary`
assert specifiers("_Complex double c;") == ["_Complex", "double"]
assert first("_Complex double c;", shapes=True)["specifiers"] == ["_Complex", "double"]
assert same("double _Complex d;\nfloat _Imaginary e;\n")
# What tree-sitter-c misreads, which the converter corrects
# 18. `typeof` of a type name
assert same("typeof(int*) x;\n") and specifiers("typeof(int) w;") == ["typeof(int)"]
# 19. Keywords and type names
path = next(PATHS)
STORE.transaction(Parse.parse("_Bool a; _Float128 b; __int128 c; size_t d; char16_t e;", path))
with STORE.transaction():
    tags = [Terms.kind_of(Objects.properties(Terms.kind_of(d)[1])["specifiers"].values[0])[0]
            for d in Objects.properties(unit(STORE, path))["body"].values]
assert tags == ["primitivetypespecifier"] * 3 + ["namedtypespecifier"] * 2
# 20. An unnamed bit-field
assert same("struct s {\n    int : 0;\n};\n")
# 21. A label ending a block
assert same("void f(void) {\n    goto l;\nl:\n}\n") and same(
    "void f(int x) {\n    switch (x) {\n        case 1:\n    }\n}\n")
# 22. GNU attributes on a null statement
fallthrough = body("switch (x) { case 1: __attribute__((fallthrough)); case 2: ; }")[0]["body"]["body"][0]
assert fallthrough["statement"] == {"kind": "AttributedStatement", "line": 2, "attributes": [
    "__attribute__((fallthrough))"], "statement": {"kind": "ExpressionStatement", "line": 2}}
# 23. `NULL`, a macro's name; `nullptr`, a constant
assert value("NULL", shapes=True) == ("variable", {"name": "NULL"}, []) and value("nullptr", shapes=True)[0] == (
    "constant")
# 24. A type name as a macro's argument
assert value("va_arg(ap, int)", shapes=True)[2][2][0] == "typeid" and value("va_arg(ap, int)") == "va_arg(ap, int)"
# 25. `defined X`, and GNU's obsolete designator
assert printed("#if defined A\n#endif\n") == "#if defined(A)\n#endif\n"
assert printed("struct S s = {x: 1};\n") == "struct S s = {.x = 1};\n"
# 26. An attribute declaration
assert kinds(parse("[[maybe_unused]];")) == ["AttributeDeclaration"] and statement("[[fallthrough]];")["kind"] == (
    "AttributeDeclaration")
# 27. `_Static_assert(x);` read as a call
assert kinds(parse("_Static_assert(1);")) == ["StaticAssertDeclaration"] and statement("_Static_assert(2);")[
    "condition"] == "2"
# 28. GNU's labels as values
assert value("&&l", shapes=True) == ("labeladdressexpression", {"label": ("identifier", {"spelling": "l"})}, [])
assert value("& &x", shapes=True)[2][0][:2] == ("unaryexpression", {"operator": "&"})  # the address of an address
assert same("int g(void) {\n    static void *table[] = {&&l, &&m};\n    goto *table[0];\nl:\nm:\n    return 0;\n}\n")
# 29. `_Pragma("...")` within an expression, read as a call
assert value('_Pragma("x")', shapes=True)[0] == "pragmaexpression" and value('f(_Pragma("y"))', shapes=True)[2][1][
    0] == "pragmaexpression"
assert statement('_Pragma("x");')["kind"] == "ExpressionStatement" and same('void f(void) {\n    _Pragma("x");\n}\n')
# 30. GNU's asm label, read as a declarator of its own, or within a function's declarator
assert first('int x asm("r1");', shapes=True)["declarators"] == [{"kind": "InitDeclarator", "declarator": 'x asm("r1")'}]
assert same('int f(void) __asm__("f2");\nint (*fp)(void) asm("fp"), *p __asm("p" "q");\n')
path = next(PATHS)
STORE.transaction(Parse.parse('int x asm("r1");', path))
with STORE.transaction():
    (declaration,) = Objects.properties(unit(STORE, path))["body"].values
    (declarator,) = Objects.properties(Terms.kind_of(declaration)[1])["declarators"].values
    named = Terms.kind_of(Objects.properties(declarator)["declarator"])[1]
    label = Objects.properties(named)["asm"]
    assert label.as_schema().name == "Expressions.C.Types.AsmLabel" and Objects.properties(label)["keyword"] == "asm"
# What tree-sitter-c reads but C rejects, which the converter keeps invalid syntax
# 31. Statements at file scope
assert parse("x = 1;\n;\n{}\nif (a) b;") == [{"kind": "Error", "line": 1, "text": "x = 1;"},
                                             {"kind": "Error", "line": 2, "text": ";"},
                                             {"kind": "Error", "line": 3, "text": "{}"},
                                             {"kind": "Error", "line": 4, "text": "if (a) b;", "body": [
                                                 {"kind": "ExpressionStatement", "line": 4, "expression": "b"}]}]
# 32. C++'s linkage specification, Microsoft's extensions
assert kinds(parse('extern "C" {\nint a;\n}\nint (__cdecl *f)(int);\nvoid __stdcall g(void);\nint * __ptr32 p;')) == [
    "Error"] * 4
assert same('extern "C" {\nint a;\n}\n')
# 33. Literals C rejects
assert [value(t, shapes=True)[0] for t in ("08", "1uu", "1.0ff", r"'\x'")] == ["error"] * 4
# 34. A conditional's closing or continuing line where none is open
assert parse("#endif\n") == [{"kind": "Error", "line": 1, "text": "#endif\n"}]
# 35. A macro's use that is no C syntax
assert kinds(parse("FOO(T) x;\nint y;")) == ["Error", "SimpleDeclaration"]
# 36. A function's definition whose declarator declares no function
assert parse("int m { return 0; }\nint (*n)[3] {}\nint o;") == [
    {"kind": "Error", "line": 1, "text": "int m { return 0; }", "body": [{"kind": "ReturnStatement", "line": 1,
                                                                      "value": "0"}]},
    {"kind": "Error", "line": 2, "text": "int (*n)[3] {}"}, {"kind": "SimpleDeclaration", "line": 3,
                                                          "specifiers": ["int"], "declarators": [
                                                              {"kind": "InitDeclarator", "declarator": "o"}]}]
assert kinds(parse("int (*f(void))(int) { return 0; }")) == ["FunctionDefinition"]  # declaring a function
# 37. GNU's asm label on a function's definition, or a second one
assert kinds(parse('int f(void) asm("g") { return 0; }\nint x asm("a") asm("b");')) == [
    "Error", "Error", "AsmDeclaration"]  # the second label, as tree-sitter recovers it, an `asm` declaration
# 38. `_Pragma` of what is no one string literal, `&&` of what is no label's name
assert [value(t, shapes=True)[:2] for t in ("_Pragma(x)", '_Pragma("a" "b")', "_Pragma()", "&&(l)")] == [
    ("error", {"text": t}) for t in ("_Pragma(x)", '_Pragma("a" "b")', "_Pragma()", "&&(l)")]
# What stays as tree-sitter-c reads it
# 39. Typedef names and other names
assert statement("T * x;")["kind"] == "SimpleDeclaration" and value("(T)*x", shapes=True)[0] == "binaryexpression"
# 40. A conditional directive that splits a construct
split = "int f(void) {\n#ifdef A\n    if (x) {\n#else\n    if (y) {\n#endif\n        z();\n    }\n}\n"
assert parse(split)[0]["kind"] == "Error" and same(split)
assert body("1.0i;")[0]["kind"] == "Error"  # GNU's imaginary constant, which tree-sitter cannot read either
# 41. Trigraphs
assert kinds(parse("int a??(3??);")) == ["Error"]
print("ok")